In [ ]:
from google.colab import files
files.upload()

In [ ]:
import os
os.makedirs("/root/.config/kaggle", exist_ok=True)
os.rename("/content/kaggle.json", "/root/.config/kaggle/kaggle.json")
os.chmod("/root/.config/kaggle/kaggle.json", 0o600)
print("Done!")

In [ ]:
!kaggle datasets download -d emmarex/plantdisease
!unzip plantdisease.zip -d /content/plantdisease
print("Dataset ready!")

In [ ]:
import os

base_path = "/content/plantdisease/plantvillage/PlantVillage"

classes = os.listdir(base_path)
print(f"Total classes: {len(classes)}")
print("\nFirst 10 classes:")
for c in classes[:10]:
    count = len(os.listdir(os.path.join(base_path, c)))
    print(f"  {c}: {count} images")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

base_path = "/content/plantdisease/plantvillage/PlantVillage"

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
classes = os.listdir(base_path)

for i, ax in enumerate(axes.flatten()):
    class_path = os.path.join(base_path, classes[i])
    img_name = os.listdir(class_path)[0]
    img = mpimg.imread(os.path.join(class_path, img_name))
    ax.imshow(img)
    ax.set_title(classes[i].replace("_", " "), fontsize=7)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

base_path = "/content/plantdisease/plantvillage/PlantVillage"

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    base_path,
    target_size=(128, 128),
    batch_size=64,
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_directory(
    base_path,
    target_size=(128, 128),
    batch_size=64,
    class_mode='categorical',
    subset='validation'
)

print(f"Training: {train_data.samples} images")
print(f"Validation: {val_data.samples} images")

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
import tensorflow as tf

base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(128, 128, 3))

# Unfreeze top 100 layers instead of 70 (slightly more capacity)
for layer in base_model.layers[:50]:
    layer.trainable = False
for layer in base_model.layers[50:]:
    layer.trainable = True

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.4)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(15, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00003),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model ready!")

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

checkpoint = ModelCheckpoint(
    'best_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=4,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    verbose=1
)

history = model.fit(
    train_data,
    epochs=15,
    validation_data=val_data,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

print("Training complete!")

In [ ]:
model.save('/content/drive/MyDrive/plant_disease_model_final.keras')
print("Final model saved!")

In [ ]:
import random

test_classes = random.sample(class_names, 4)

for test_class in test_classes:
    test_folder = os.path.join(base_path, test_class)
    test_img = random.choice(os.listdir(test_folder))
    test_img_path = os.path.join(test_folder, test_img)

    print(f"\n--- Actual: {test_class} ---")
    predict_disease(test_img_path)

In [ ]:
!pip install streamlit -q
print("Streamlit installed!")

In [ ]:
%%writefile app.py
import streamlit as st
import tensorflow as tf
import numpy as np
from PIL import Image

@st.cache_resource
def load_model():
    return tf.keras.models.load_model('plant_disease_model_final.keras')

model = load_model()

class_names = [
    'Pepper__bell___Bacterial_spot',
    'Pepper__bell___healthy',
    'Potato___Early_blight',
    'Potato___Late_blight',
    'Potato___healthy',
    'Tomato_Bacterial_spot',
    'Tomato_Early_blight',
    'Tomato_Late_blight',
    'Tomato_Leaf_Mold',
    'Tomato_Septoria_leaf_spot',
    'Tomato_Spider_mites_Two_spotted_spider_mite',
    'Tomato_Target_Spot',
    'Tomato_Tomato_mosaic_virus',
    'Tomato_Tomato_YellowLeaf_Curl_Virus',
    'Tomato_healthy'
]

st.set_page_config(page_title="Crop Disease Detector", page_icon="🌱")
st.title("🌱 Crop Disease Detection AI")
st.write("Upload a leaf image (Tomato, Potato, or Pepper) to detect disease")

uploaded_file = st.file_uploader("Choose an image...", type=['jpg', 'jpeg', 'png'])

if uploaded_file is not None:
    img = Image.open(uploaded_file).convert('RGB')
    st.image(img, caption='Uploaded Image', use_column_width=True)

    img_resized = img.resize((128, 128))
    img_array = np.array(img_resized) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    with st.spinner('Analyzing...'):
        predictions = model.predict(img_array)

    predicted_class = class_names[np.argmax(predictions)]
    confidence = np.max(predictions) * 100

    clean_name = predicted_class.replace('___', ' - ').replace('_', ' ')

    if confidence > 70:
        st.success(f"**Detected:** {clean_name}")
    else:
        st.warning(f"**Detected:** {clean_name} (low confidence)")

    st.metric("Confidence", f"{confidence:.1f}%")

In [ ]:
!pip install pyngrok -q
print("ngrok installed!")

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3FNtNr6ITEfVU6bMPdCx0VMbv6v_3WMECEEjuWFqtZ9PHuaXZ")
print("Token set!")

In [ ]:
!cp /content/drive/MyDrive/plant_disease_model_final.keras /content/plant_disease_model_final.keras
print("Model copied!")

In [ ]:
import subprocess
import time

# Kill any existing streamlit processes
!pkill -f streamlit

# Start streamlit in background
process = subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501'])

time.sleep(5)

from pyngrok import ngrok
public_url = ngrok.connect(8501)
print(f"Your app is live at: {public_url}")

In [ ]:
# Verify model performance
val_loss, val_accuracy = model.evaluate(val_data)
print(f"Validation Accuracy: {val_accuracy*100:.2f}%")
print(f"Validation Loss: {val_loss:.4f}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
files = os.listdir('/content/drive/MyDrive')
print(files)

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt
import os

# Load the saved model
model = tf.keras.models.load_model('/content/drive/MyDrive/plant_disease_model_v2.keras')

# Download dataset again if not already in this session
if not os.path.exists('/content/plantdisease'):
    !kaggle datasets download -d emmarex/plantdisease
    !unzip -q plantdisease.zip -d /content/plantdisease

base_path = "/content/plantdisease/plantvillage/PlantVillage"
class_names = sorted(os.listdir(base_path))

def predict_disease(img_path):
    img = image.load_img(img_path, target_size=(128, 128))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = img_array / 255.0

    predictions = model.predict(img_array)
    predicted_class = class_names[np.argmax(predictions)]
    confidence = np.max(predictions) * 100

    plt.imshow(img)
    plt.title(f"Prediction: {predicted_class}\nConfidence: {confidence:.2f}%")
    plt.axis('off')
    plt.show()

    print(f"Disease: {predicted_class}")
    print(f"Confidence: {confidence:.2f}%")

# Test on a sample image
test_class = class_names[5]  # pick any class
test_folder = os.path.join(base_path, test_class)
test_img = os.listdir(test_folder)[0]
test_img_path = os.path.join(test_folder, test_img)

print(f"Testing on: {test_class}")
predict_disease(test_img_path)

In [ ]:
import random

# Test on 4 random images from different classes
test_classes = random.sample(class_names, 4)

for test_class in test_classes:
    test_folder = os.path.join(base_path, test_class)
    test_img = random.choice(os.listdir(test_folder))
    test_img_path = os.path.join(test_folder, test_img)

    print(f"\n--- Actual: {test_class} ---")
    predict_disease(test_img_path)

In [ ]:
!pip install streamlit -q
print("Streamlit installed!")